# ITRI external validation: BRITS and SAITS

Choose an A100 GPU runtime (high RAM optional), then Run all. Upload **two files, one at a time**, when prompted: first `ITRI_external_PRIVATE_data_snapshot.zip`, then `ITRI_classical_full_results.zip`. They can be in different folders. SHA-256 verifies both. Authorize Google Drive mount; only metric checkpoints go there, not vendor OHLCV.

The frozen full grid requires 11,016 neural fits and may exceed one Colab session. If disconnected, open the same notebook again, Run all, reupload the same two ZIPs, and authorize Drive. Verified condition checkpoints are reused. A final download is offered only after all neural and combined checks pass. Pilot results are not scientific results.

---

**Devam edebilir surum.** Tamamlanmis kosullar Google Drive'da tutulur ve yeniden calistirildiginda atlanir. Devam anahtari yalnizca protokolu kapsar; GPU modeli, surucu ve torch surumu anahtarin disindadir ve `environment_history.json` dosyasina kaydedilir. Oturum kesilirse **Run all** demeniz yeterlidir: iki ZIP tekrar istenir, Drive tekrar baglanir, hesaplama kaldigi yerden surer.


In [ ]:
%pip -q install pypots==1.5 numpy==2.1.3 pandas==2.2.3 scipy==1.18.1 scikit-learn==1.6.1

In [ ]:
from pathlib import Path
work_dir = Path('/content/itri_neural')
work_dir.mkdir(parents=True, exist_ok=True)
print('Working directory:', work_dir)

In [ ]:
%%writefile /content/itri_neural/masking.py
"""Controlled missing-run masks for a new, separately reported validation study.

The archived ITRI experiment permits adjacent sampled segments. This module
does not alter those historical masks or results. It defines strict runs for a
future experiment in which the requested block length has a literal meaning.
"""

from __future__ import annotations

import numpy as np


def run_lengths(mask: np.ndarray) -> np.ndarray:
    """Return every maximal missing-run length, separately for each feature."""
    if mask.ndim != 2 or mask.dtype != np.bool_:
        raise ValueError("mask must be a two-dimensional boolean array")
    lengths: list[int] = []
    for feature in range(mask.shape[1]):
        edges = np.flatnonzero(np.diff(np.r_[False, mask[:, feature], False]))
        lengths.extend((edges[1::2] - edges[::2]).tolist())
    return np.asarray(lengths, dtype=int)


def strict_block_mask(
    shape: tuple[int, int], rate: float, block_length: int, seed: int,
    edge_guard: int = 10,
) -> np.ndarray:
    """Place nonadjacent runs of length ``block_length`` at an exact cell rate.

    Every feature receives one full run before the remaining runs are sampled,
    so feature-wise errors are defined for every condition. A final shorter
    run is allowed only when the rounded target cell count is not divisible
    by the requested block length. One observed row separates every pair of
    runs within a feature. Other features may be missing on the same dates.
    """
    rows, features = shape
    if rows <= 2 * edge_guard + block_length or features < 1:
        raise ValueError("not enough rows/features for the block and guard")
    if not 0 < rate < 1 or block_length < 1 or edge_guard < 0:
        raise ValueError("invalid rate, block length, or edge guard")
    eligible_rows = rows - 2 * edge_guard
    target = max(1, round(rate * eligible_rows * features))
    if target < features * block_length:
        raise ValueError("target too small for one full masked run per feature")
    rng = np.random.default_rng(seed)
    mask = np.zeros(shape, dtype=bool)
    # The endpoint is a five-feature macro mean, so a zero-mask feature is
    # invalid. Reserve exactly one nominal-length run per feature first.
    for feature in rng.permutation(features):
        start = int(rng.integers(edge_guard,
                                 rows - edge_guard - block_length + 1))
        mask[start:start + block_length, feature] = True
    remaining = target - features * block_length
    attempts = 0
    max_attempts = 1000 * target

    while remaining:
        length = min(block_length, remaining)
        feature = int(rng.integers(features))
        start = int(rng.integers(edge_guard, rows - edge_guard - length + 1))
        neighborhood = mask[max(edge_guard, start - 1):
                            min(rows - edge_guard, start + length + 1), feature]
        if not neighborhood.any():
            mask[start:start + length, feature] = True
            remaining -= length
        attempts += 1
        if attempts >= max_attempts:
            raise RuntimeError("could not place the target runs without adjacency")

    lengths = run_lengths(mask)
    if (int(mask.sum()) != target or lengths.max() > block_length or
            not np.all(mask.sum(axis=0) >= block_length)):
        raise AssertionError("strict-block mask invariant failed")
    if np.any(mask[:edge_guard]) or np.any(mask[rows - edge_guard:]):
        raise AssertionError("strict-block mask crossed the edge guard")
    return mask


In [ ]:
%%writefile /content/itri_neural/run_classical_benchmark.py
#!/usr/bin/env python3
"""Frozen OHLCV external-validation benchmark, classical comparators.

This program reads the PRIVATE vendor snapshot ZIP without extracting it. It
never writes OHLCV values to the public result directory. All methods receive
the exact same mask per asset/fold/condition. Outputs are checkpointed by
asset/fold. Use --pilot for a small smoke test; never report pilot results as
the full prespecified experiment.
"""

from __future__ import annotations

import argparse
import hashlib
import io
import json
import os
import platform
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.interpolate import PchipInterpolator
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.exceptions import ConvergenceWarning

from masking import run_lengths, strict_block_mask


FEATURES = ("open", "high", "low", "close", "volume")
METHODS = ("linear", "pchip", "knn5", "chained_ridge", "itri_w1_p5")
FOLDS = {
    "A": {"train": (2015, 2018), "val": (2019, 2019), "test": (2020, 2021)},
    "B": {"train": (2015, 2020), "val": (2021, 2021), "test": (2022, 2023)},
    "C": {"train": (2015, 2022), "val": (2023, 2023), "test": (2024, 2025)},
}
BLOCKS = (1, 2, 3, 5, 10, 20)
RATES = (0.10, 0.20, 0.30)
SEEDS = (11, 23, 37)
EDGE_GUARD = 20


def sha256(blob: bytes) -> str:
    return hashlib.sha256(blob).hexdigest()


def stable_mask_seed(manifest_sha: str, ticker: str, fold: str,
                     partition: str, block: int, rate: float,
                     repetition_seed: int) -> int:
    key = (f"{manifest_sha}|{ticker}|{fold}|{partition}|"
           f"{block}|{rate:.2f}|{repetition_seed}")
    return int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big")


def split_frame(frame: pd.DataFrame, fold: str) -> dict[str, pd.DataFrame]:
    parts = {}
    for name, (start_year, end_year) in FOLDS[fold].items():
        subset = frame.loc[(frame.index.year >= start_year) &
                           (frame.index.year <= end_year), list(FEATURES)]
        if subset.empty:
            raise ValueError(f"empty {fold}/{name} partition")
        parts[name] = subset.copy()
    return parts


def train_observed_scaler(train: np.ndarray, hidden: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if train.shape != hidden.shape:
        raise ValueError("training array/mask shape mismatch")
    means, scales = [], []
    for feature in range(train.shape[1]):
        observed = train[~hidden[:, feature], feature]
        if observed.size < 2 or not np.isfinite(observed).all():
            raise ValueError(f"insufficient finite training data for feature {feature}")
        mean = float(observed.mean())
        scale = float(observed.std(ddof=0))
        if scale <= 1e-12:
            raise ValueError(f"near-zero training-observed std for feature {feature}")
        means.append(mean)
        scales.append(scale)
    return np.asarray(means), np.asarray(scales)


def corrupt(values: np.ndarray, hidden: np.ndarray) -> np.ndarray:
    output = values.copy()
    output[hidden] = np.nan
    return output


def linear_fill(corrupt_values: np.ndarray) -> np.ndarray:
    return pd.DataFrame(corrupt_values).interpolate(
        method="linear", limit_direction="both").ffill().bfill().to_numpy(float)


def pchip_fill(corrupt_values: np.ndarray) -> np.ndarray:
    out = corrupt_values.copy()
    x = np.arange(len(out))
    for feature in range(out.shape[1]):
        seen = np.isfinite(out[:, feature])
        if seen.sum() < 2:
            raise ValueError("PCHIP needs at least two observed cells")
        curve = PchipInterpolator(x[seen], out[seen, feature], extrapolate=True)
        out[~seen, feature] = curve(x[~seen])
    return out


class ITRIImputer:
    """Exact w=1, p=5 temporal-regression logic from the archived implementation.

    Only the panel and observed-only scaling differ from the original study.
    The base model has not been modified to chase external test results.
    """

    def __init__(self) -> None:
        self.window = 1
        self.models_: list[LinearRegression | None] = []

    def temporal_matrix(self, col: np.ndarray) -> np.ndarray:
        z = np.column_stack((np.roll(col, 1), np.roll(col, -1)))
        z[0, :] = np.nan
        z[-1, :] = np.nan
        return z

    def fit(self, train_corrupt: np.ndarray) -> "ITRIImputer":
        missing = np.isnan(train_corrupt)
        completed = linear_fill(train_corrupt)
        self.models_ = []
        for feature in range(train_corrupt.shape[1]):
            z = self.temporal_matrix(completed[:, feature])
            good = (~missing[:, feature]) & np.isfinite(z).all(axis=1)
            if int(good.sum()) < 4:
                self.models_.append(None)
                continue
            model = LinearRegression()
            model.fit(z[good], completed[good, feature])
            self.models_.append(model)
        return self

    def transform(self, test_corrupt: np.ndarray) -> np.ndarray:
        missing = np.isnan(test_corrupt)
        observed = ~missing
        completed = linear_fill(test_corrupt)
        for _ in range(5):
            for feature, model in enumerate(self.models_):
                if model is None:
                    continue
                z = self.temporal_matrix(completed[:, feature])
                rows = missing[:, feature] & np.isfinite(z).all(axis=1)
                if rows.any():
                    completed[rows, feature] = model.predict(z[rows])
            completed[observed] = test_corrupt[observed]
        return completed


def run_methods(train_corrupt: np.ndarray, test_corrupt: np.ndarray,
                random_seed: int) -> dict[str, tuple[np.ndarray, float, int]]:
    outputs = {}
    start = time.perf_counter()
    outputs["linear"] = (linear_fill(test_corrupt), time.perf_counter() - start, 0)
    start = time.perf_counter()
    outputs["pchip"] = (pchip_fill(test_corrupt), time.perf_counter() - start, 0)
    start = time.perf_counter()
    knn = KNNImputer(n_neighbors=5, weights="distance")
    knn.fit(train_corrupt)
    outputs["knn5"] = (knn.transform(test_corrupt), time.perf_counter() - start, 0)
    start = time.perf_counter()
    chained = IterativeImputer(
        estimator=Ridge(alpha=1.0), max_iter=10, tol=1e-3,
        initial_strategy="mean", imputation_order="ascending",
        sample_posterior=False, random_state=random_seed,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        chained.fit(train_corrupt)
        chained_estimate = chained.transform(test_corrupt)
    nonconvergence = sum(issubclass(note.category, ConvergenceWarning)
                         for note in caught)
    outputs["chained_ridge"] = (chained_estimate,
                                 time.perf_counter() - start, nonconvergence)
    start = time.perf_counter()
    temporal = ITRIImputer().fit(train_corrupt)
    outputs["itri_w1_p5"] = (temporal.transform(test_corrupt),
                               time.perf_counter() - start, 0)
    if tuple(outputs) != METHODS:
        raise AssertionError("method registry/order mismatch")
    for method, (estimate, _, _) in outputs.items():
        if estimate.shape != test_corrupt.shape or not np.isfinite(estimate).all():
            raise ValueError(f"{method} produced invalid estimates")
        observed = np.isfinite(test_corrupt)
        if not np.allclose(estimate[observed], test_corrupt[observed],
                           atol=1e-9, rtol=1e-9):
            raise ValueError(f"{method} changed originally observed cells")
    return outputs


def score_features(truth_raw: np.ndarray, estimate_scaled: np.ndarray,
                   hidden: np.ndarray, mean: np.ndarray, scale: np.ndarray,
                   common: dict, elapsed: float,
                   convergence_warning_count: int) -> list[dict]:
    estimate_raw = estimate_scaled * scale + mean
    results = []
    for feature_index, feature in enumerate(FEATURES):
        selected = hidden[:, feature_index]
        count = int(selected.sum())
        if count < 1:
            raise ValueError(f"no masked cells for {feature}")
        error = estimate_raw[selected, feature_index] - truth_raw[selected, feature_index]
        mae = float(np.abs(error).mean())
        rmse = float(np.sqrt(np.square(error).mean()))
        results.append({
            **common,
            "feature": feature,
            "masked_cells": count,
            "mae_raw": mae,
            "rmse_raw": rmse,
            "mae_trainstd": mae / scale[feature_index],
            "rmse_trainstd": rmse / scale[feature_index],
            "fit_and_impute_seconds": elapsed,
            "convergence_warning_count": convergence_warning_count,
        })
    return results


def load_verified_asset(archive: zipfile.ZipFile, record: dict) -> pd.DataFrame:
    name = f"clean_vendor_cache_do_not_publish/{record['ticker']}_2015-01-01_2026-01-01.csv"
    blob = archive.read(name)
    if sha256(blob) != record["clean_sha256"]:
        raise ValueError(f"private data hash mismatch: {record['ticker']}")
    frame = pd.read_csv(io.BytesIO(blob), index_col=0,
                        parse_dates=True, float_precision="round_trip")
    if frame.columns.tolist() != list(FEATURES) or len(frame) != record["row_count"]:
        raise ValueError(f"private data shape mismatch: {record['ticker']}")
    if not np.isfinite(frame.to_numpy(float)).all():
        raise ValueError(f"nonfinite data: {record['ticker']}")
    return frame


def run(snapshot: Path, output_dir: Path, pilot: bool = False) -> None:
    with zipfile.ZipFile(snapshot) as archive:
        manifest_blob = archive.read("asset_universe_v1.json")
        receipt_blob = archive.read("acquisition_qc_receipt.json")
        receipt = json.loads(receipt_blob)
        manifest = json.loads(manifest_blob)
        manifest_sha = sha256(manifest_blob)
        if receipt["manifest_sha256"] != manifest_sha:
            raise ValueError("snapshot manifest/receipt SHA mismatch")
        if receipt["eligible_count"] != 34 or receipt["excluded_count"]:
            raise ValueError("unexpected eligibility count; revise only by protocol amendment")
        if [x["ticker"] for x in manifest["assets"]] != [
                x["ticker"] for x in receipt["assets"]]:
            raise ValueError("manifest/receipt asset order mismatch")

        output_dir.mkdir(parents=True, exist_ok=True)
        shard_dir = output_dir / "shards"
        shard_dir.mkdir(exist_ok=True)
        versions = {"python": platform.python_version(), "numpy": np.__version__,
                    "pandas": pd.__version__, "scipy": scipy.__version__,
                    "sklearn": sklearn.__version__}
        run_fingerprint = sha256(
            Path(__file__).read_bytes() + Path(__file__).with_name("masking.py").read_bytes()
            + manifest_blob + receipt_blob + str(pilot).encode()
            + json.dumps(versions, sort_keys=True).encode())
        run_meta = {
            "run_fingerprint": run_fingerprint,
            "protocol": manifest["protocol_version"],
            "manifest_sha256": manifest_sha,
            "private_zip_sha256": sha256(snapshot.read_bytes()),
            "pilot_not_publishable": pilot,
            "methods": METHODS,
            "folds": FOLDS,
            "blocks": BLOCKS,
            "rates": RATES,
            "seeds": SEEDS,
            "edge_guard": EDGE_GUARD,
            "primary_panel": FEATURES,
            "primary_endpoint": "macro mean across 5 feature-wise RMSE/training-observed-SD",
            "versions": versions,
        }
        (output_dir / "run_metadata.json").write_text(
            json.dumps(run_meta, indent=2), encoding="utf-8")
        records = receipt["assets"][:1] if pilot else receipt["assets"]
        folds = ("A",) if pilot else tuple(FOLDS)
        conditions = [(b, r, s) for b in BLOCKS for r in RATES for s in SEEDS]
        if pilot:
            conditions = conditions[:2]
        for asset_index, record in enumerate(records, start=1):
            ticker = record["ticker"]
            frame = load_verified_asset(archive, record)
            for fold in folds:
                shard_path = shard_dir / f"{ticker}_{fold}.csv"
                meta_path = shard_dir / f"{ticker}_{fold}.meta.json"
                expected_rows = len(conditions) * len(METHODS) * len(FEATURES)
                if shard_path.exists() and meta_path.exists():
                    sidecar = json.loads(meta_path.read_text())
                    if (sidecar.get("run_fingerprint") == run_fingerprint and
                            sidecar.get("expected_rows") == expected_rows and
                            sidecar.get("csv_sha256") == sha256(shard_path.read_bytes())):
                        print(f"[{asset_index}/{len(records)}] {ticker}/{fold}: verified checkpoint")
                        continue
                partitions = split_frame(frame, fold)
                train = partitions["train"].to_numpy(float)
                test = partitions["test"].to_numpy(float)
                rows = []
                for condition_index, (block, rate, repetition_seed) in enumerate(conditions, 1):
                    train_seed = stable_mask_seed(manifest_sha, ticker, fold, "train",
                                                  block, rate, repetition_seed)
                    test_seed = stable_mask_seed(manifest_sha, ticker, fold, "test",
                                                 block, rate, repetition_seed)
                    train_hidden = strict_block_mask(train.shape, rate, block,
                                                     train_seed, EDGE_GUARD)
                    test_hidden = strict_block_mask(test.shape, rate, block,
                                                    test_seed, EDGE_GUARD)
                    mean, scale = train_observed_scaler(train, train_hidden)
                    train_scaled = (train - mean) / scale
                    test_scaled = (test - mean) / scale
                    estimates = run_methods(corrupt(train_scaled, train_hidden),
                                            corrupt(test_scaled, test_hidden),
                                            repetition_seed)
                    lengths = run_lengths(test_hidden)
                    common = {
                        "ticker": ticker, "country": record["country"],
                        "region": record["region"], "fold": fold,
                        "test_start_year": FOLDS[fold]["test"][0],
                        "test_end_year": FOLDS[fold]["test"][1],
                        "block_length": block, "rate": rate,
                        "seed": repetition_seed,
                        "train_mask_seed": train_seed,
                        "test_mask_seed": test_seed,
                        "actual_max_test_run": int(lengths.max()),
                    }
                    if common["actual_max_test_run"] > block:
                        raise AssertionError("mask-length invariant violated")
                    for method, (estimate, elapsed, warning_count) in estimates.items():
                        rows.extend(score_features(test, estimate, test_hidden,
                                                   mean, scale,
                                                   {**common, "method": method},
                                                   elapsed, warning_count))
                    if condition_index % 10 == 0 or condition_index == len(conditions):
                        print(f"[{asset_index}/{len(records)}] {ticker}/{fold}: "
                              f"{condition_index}/{len(conditions)} conditions")
                if len(rows) != expected_rows:
                    raise AssertionError("missing or duplicate method-feature rows")
                temporary = shard_path.with_suffix(".csv.tmp")
                pd.DataFrame(rows).to_csv(temporary, index=False, float_format="%.17g")
                os.replace(temporary, shard_path)
                sidecar = {
                    "run_fingerprint": run_fingerprint,
                    "expected_rows": expected_rows,
                    "csv_sha256": sha256(shard_path.read_bytes()),
                    "pilot_not_publishable": pilot,
                }
                meta_path.write_text(json.dumps(sidecar, indent=2), encoding="utf-8")
        expected_shards = len(records) * len(folds)
        verified_shards = list(shard_dir.glob("*.meta.json"))
        if len(verified_shards) < expected_shards:
            raise AssertionError("run ended with missing checkpoints")
        (output_dir / "COMPLETE.marker").write_text(
            json.dumps({"expected_shards": expected_shards,
                        "expected_rows": expected_shards * len(conditions)
                        * len(METHODS) * len(FEATURES),
                        "pilot_not_publishable": pilot,
                        "run_fingerprint": run_fingerprint}, indent=2),
            encoding="utf-8")
        print("COMPLETE", output_dir)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--snapshot", type=Path, required=True,
                        help="Private data snapshot ZIP from acquisition notebook")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--pilot", action="store_true",
                        help="Smoke test only: first asset/fold/two masks; not publishable")
    args = parser.parse_args()
    run(args.snapshot, args.out, pilot=args.pilot)


In [ ]:
%%writefile /content/itri_neural/run_neural_benchmark.py
#!/usr/bin/env python3
"""BRITS/SAITS external validation on the frozen classical benchmark masks.

One neural model is fitted for each asset, fold, block length, missing rate,
mask seed and method. Only the corresponding masked training partition is
used for fitting; the separate validation partition selects the epoch. Test
truth is used only by score_features after inference. Checkpoints contain
metrics and metadata, never raw OHLCV or model weights.
"""

from __future__ import annotations

import argparse
import hashlib
import io
import json
import logging
import os
import platform
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pypots
import pygrinder
import scipy
import sklearn
import torch
from pypots.imputation import BRITS, SAITS
from pypots.utils.logging import logger as pypots_logger

from masking import run_lengths, strict_block_mask
from run_classical_benchmark import (
    BLOCKS, EDGE_GUARD, FEATURES, FOLDS, RATES, SEEDS, METHODS,
    corrupt, load_verified_asset, run_methods, score_features, sha256, split_frame,
    stable_mask_seed, train_observed_scaler,
)


NEURAL_METHODS = ("brits", "saits")
N_STEPS = 64
STRIDE = 16
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 5
BRITS_HIDDEN = 64
SAITS_CONFIG = {"n_layers": 2, "d_model": 64, "n_heads": 4,
                "d_k": 16, "d_v": 16, "d_ffn": 128, "dropout": 0.1}


class _NoSavingPathWarning(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        return "saving_path not given" not in record.getMessage()


pypots_logger.addFilter(_NoSavingPathWarning())


def window_starts(length: int, n_steps: int = N_STEPS,
                  stride: int = STRIDE) -> list[int]:
    if length < n_steps or n_steps < 2 or stride < 1:
        raise ValueError("partition is too short for the frozen neural window")
    starts = list(range(0, length - n_steps + 1, stride))
    if starts[-1] != length - n_steps:
        starts.append(length - n_steps)
    return starts


def as_windows(values: np.ndarray, starts: list[int],
               n_steps: int = N_STEPS) -> np.ndarray:
    return np.stack([values[start:start + n_steps] for start in starts]).astype(
        np.float32, copy=False)


def fold_windows(predictions: np.ndarray, starts: list[int],
                 observed: np.ndarray) -> np.ndarray:
    """Average overlapping predictions; restore every originally observed cell."""
    if predictions.shape != (len(starts), N_STEPS, observed.shape[1]):
        raise ValueError("neural prediction shape mismatch")
    sums = np.zeros(observed.shape, dtype=np.float64)
    counts = np.zeros(observed.shape[0], dtype=np.int32)
    for prediction, start in zip(predictions, starts):
        sums[start:start + N_STEPS] += prediction
        counts[start:start + N_STEPS] += 1
    if np.any(counts == 0):
        raise AssertionError("test window coverage is incomplete")
    completed = sums / counts[:, None]
    originally_observed = np.isfinite(observed)
    completed[originally_observed] = observed[originally_observed]
    if not np.isfinite(completed).all():
        raise ValueError("neural model produced non-finite estimates")
    return completed


def set_training_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def training_seed(manifest_sha: str, ticker: str, fold: str,
                  block: int, rate: float, repetition_seed: int,
                  method: str) -> int:
    key = (f"{manifest_sha}|{ticker}|{fold}|{block}|{rate:.2f}|"
           f"{repetition_seed}|{method}|neural-v1")
    return int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big")


def fit_and_predict(method: str, train_input: np.ndarray,
                    val_input: np.ndarray, val_truth: np.ndarray,
                    test_input: np.ndarray, seed: int,
                    device: str, epochs: int = EPOCHS) -> tuple[np.ndarray, np.ndarray, float]:
    set_training_seed(seed)
    train_starts = window_starts(len(train_input))
    val_starts = window_starts(len(val_input))
    test_starts = window_starts(len(test_input))
    train_set = {"X": as_windows(train_input, train_starts)}
    val_set = {"X": as_windows(val_input, val_starts),
               "X_ori": as_windows(val_truth, val_starts)}
    test_set = {"X": as_windows(test_input, test_starts)}
    common = {"n_steps": N_STEPS, "n_features": len(FEATURES),
              "batch_size": BATCH_SIZE, "epochs": epochs,
              "patience": min(PATIENCE, epochs), "device": device,
              "model_saving_strategy": None, "verbose": False}
    if method == "brits":
        model = BRITS(rnn_hidden_size=BRITS_HIDDEN, **common)
    elif method == "saits":
        model = SAITS(**SAITS_CONFIG, **common)
    else:
        raise ValueError(f"unknown neural method: {method}")
    started = time.perf_counter()
    model.fit(train_set, val_set)
    val_predictions = np.asarray(model.predict({"X": val_set["X"]})["imputation"])
    test_predictions = np.asarray(model.predict(test_set)["imputation"])
    val_completed = fold_windows(val_predictions, val_starts, val_input)
    test_completed = fold_windows(test_predictions, test_starts, test_input)
    elapsed = time.perf_counter() - started
    del model
    if device.startswith("cuda"):
        torch.cuda.empty_cache()
    return val_completed, test_completed, elapsed


def verified_classical_shard(archive: zipfile.ZipFile, ticker: str,
                             fold: str, expected_fingerprint: str) -> dict:
    name = f"results/shards/{ticker}_{fold}.csv"
    sidecar = json.loads(archive.read(name.replace(".csv", ".meta.json")))
    blob = archive.read(name)
    if (sidecar["csv_sha256"] != sha256(blob) or
            sidecar["run_fingerprint"] != expected_fingerprint):
        raise ValueError(f"classical checkpoint checksum mismatch: {ticker}/{fold}")
    # Pandas' default parser can turn the archived 0.29999999999999999 into
    # 0.2999999999999999, which does not equal the prespecified Python 0.3.
    frame = pd.read_csv(io.BytesIO(blob), float_precision="round_trip")
    reference = frame.loc[(frame.method == "linear") & (frame.feature == "open")]
    if len(reference) != len(BLOCKS) * len(RATES) * len(SEEDS):
        raise ValueError(f"incomplete classical condition grid: {ticker}/{fold}")
    return {(int(row.block_length), float(row.rate), int(row.seed)):
            (int(row.train_mask_seed), int(row.test_mask_seed))
            for row in reference.itertuples()}


def migrate_checkpoints(output_dir: Path, fingerprint: str,
                        allowed_previous: set[str]) -> dict:
    """Relabel intact shards written under an earlier environment fingerprint.

    This is a one-time migration for checkpoints produced before the resume key
    was made hardware independent. Only a fingerprint listed in
    ``allowed_previous`` is accepted, so a genuine protocol change can never be
    laundered into the current run. Only the provenance key is rewritten. Every
    CSV is re-hashed first, so a shard is kept only when its bytes still match
    the digest recorded beside it. Anything that fails verification is deleted
    and recomputed.
    """
    migrated = 0
    dropped = 0
    superseded: set[str] = set()
    for directory, expected in ((output_dir / "shards", 10),
                                (output_dir / "validation_shards", 35)):
        if not directory.is_dir():
            continue
        for meta_path in sorted(directory.glob("*.meta.json")):
            csv_path = meta_path.with_name(meta_path.name.replace(".meta.json", ".csv"))
            if not csv_path.exists():
                meta_path.unlink(missing_ok=True)
                dropped += 1
                continue
            try:
                sidecar = json.loads(meta_path.read_text())
            except json.JSONDecodeError:
                meta_path.unlink(missing_ok=True)
                csv_path.unlink(missing_ok=True)
                dropped += 1
                continue
            if sidecar.get("run_fingerprint") == fingerprint:
                continue
            if sidecar.get("run_fingerprint") not in allowed_previous:
                continue
            if (sidecar.get("expected_rows") != expected or
                    sidecar.get("csv_sha256") != sha256(csv_path.read_bytes())):
                meta_path.unlink(missing_ok=True)
                csv_path.unlink(missing_ok=True)
                dropped += 1
                continue
            previous = str(sidecar.get("run_fingerprint"))
            superseded.add(previous)
            sidecar.setdefault("migrated_from", []).append(previous)
            sidecar["run_fingerprint"] = fingerprint
            atomic_write_bytes(meta_path, json.dumps(sidecar, indent=2).encode())
            migrated += 1
    return {"migrated": migrated, "dropped": dropped,
            "superseded_fingerprints": sorted(superseded)}


def atomic_write_bytes(path: Path, data: bytes) -> None:
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_bytes(data)
    os.replace(temporary, path)


def run(snapshot: Path, classical_zip: Path, output_dir: Path,
        pilot: bool = False) -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    with zipfile.ZipFile(snapshot) as private, zipfile.ZipFile(classical_zip) as classical:
        manifest_blob = private.read("asset_universe_v1.json")
        receipt_blob = private.read("acquisition_qc_receipt.json")
        manifest_sha = sha256(manifest_blob)
        receipt = json.loads(receipt_blob)
        classical_meta = json.loads(classical.read("results/run_metadata.json"))
        classical_marker = json.loads(classical.read("results/COMPLETE.marker"))
        if (receipt["manifest_sha256"] != manifest_sha or
                classical_meta["manifest_sha256"] != manifest_sha or
                classical_meta["private_zip_sha256"] != sha256(snapshot.read_bytes()) or
                classical_marker["expected_shards"] != 102 or
                classical_marker["pilot_not_publishable"] or
                classical_marker["run_fingerprint"] != classical_meta["run_fingerprint"]):
            raise ValueError("private/classical inputs are incomplete or do not match")
        if (receipt["eligible_count"] != 34 or receipt["excluded_count"] or
                tuple(classical_meta["methods"]) != METHODS or
                classical_meta["edge_guard"] != EDGE_GUARD):
            raise ValueError("classical protocol or panel differs from the frozen run")

        versions = {"python": platform.python_version(),
                    "numpy": np.__version__, "pandas": pd.__version__,
                    "scipy": scipy.__version__, "sklearn": sklearn.__version__,
                    "torch": torch.__version__, "pypots": pypots.__version__,
                    "pygrinder": pygrinder.__version__,
                    "torch_cuda": torch.version.cuda,
                    "cudnn": torch.backends.cudnn.version()}
        hardware = {"device": device,
                    "gpu_name": torch.cuda.get_device_name(0)
                    if device == "cuda" else None}
        config = {"window_length": N_STEPS, "window_stride": STRIDE,
                  "batch_size": BATCH_SIZE, "epochs_max": EPOCHS,
                  "patience": PATIENCE, "brits_hidden": BRITS_HIDDEN,
                  "saits": SAITS_CONFIG, "methods": NEURAL_METHODS,
                  "device": device, "hardware": hardware}
        # The resume key covers the protocol only. Accelerator model, driver
        # stack and framework build are recorded in the metadata but excluded,
        # so a run may legitimately span several sessions and GPUs.
        protocol_config = {"window_length": N_STEPS, "window_stride": STRIDE,
                           "batch_size": BATCH_SIZE, "epochs_max": EPOCHS,
                           "patience": PATIENCE, "brits_hidden": BRITS_HIDDEN,
                           "saits": SAITS_CONFIG, "methods": NEURAL_METHODS}
        protocol_versions = {key: versions[key] for key in
                             ("numpy", "pandas", "scipy", "sklearn",
                              "pypots", "pygrinder")}
        fingerprint = sha256(
            Path(__file__).read_bytes() + Path(__file__).with_name("masking.py").read_bytes()
            + Path(__file__).with_name("run_classical_benchmark.py").read_bytes()
            + manifest_blob + receipt_blob + classical_marker["run_fingerprint"].encode()
            + json.dumps(protocol_config, sort_keys=True).encode()
            + json.dumps(protocol_versions, sort_keys=True).encode()
            + str(pilot).encode())
        output_dir.mkdir(parents=True, exist_ok=True)
        shard_dir = output_dir / "shards"
        shard_dir.mkdir(exist_ok=True)
        validation_dir = output_dir / "validation_shards"
        validation_dir.mkdir(exist_ok=True)
        meta_path = output_dir / "run_metadata.json"
        previous_meta = (json.loads(meta_path.read_text())
                         if meta_path.exists() else {})
        previous_key = previous_meta.get("run_fingerprint")
        legacy = bool(previous_meta) and "resume_key_scope" not in previous_meta
        migration = {"migrated": 0, "dropped": 0, "superseded_fingerprints": []}
        if legacy and previous_key and previous_key != fingerprint:
            # One-time move from the old hardware-dependent key to the new one.
            migration = migrate_checkpoints(output_dir, fingerprint,
                                            {previous_key})
            print(f"Checkpoint migration: {migration['migrated']} relabelled, "
                  f"{migration['dropped']} discarded as unverifiable",
                  flush=True)
        elif previous_key and previous_key != fingerprint:
            changed = {k: (previous_meta.get("protocol_versions", {}).get(k), v)
                       for k, v in protocol_versions.items()
                       if previous_meta.get("protocol_versions", {}).get(k) != v}
            raise SystemExit(
                "Protokol anahtari degisti, tamamlanmis shard'lar bu kosuya "
                "dahil edilemez. Farkli kutuphaneler: " + json.dumps(changed)
                + ". Ya ortami eski surumlere sabitleyin ya da sonuc klasorunu "
                "yeni bir isimle bastan baslatin.")

        history_path = output_dir / "environment_history.json"
        history = (json.loads(history_path.read_text())
                   if history_path.exists() else [])
        history.append({"started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ",
                                                     time.gmtime()),
                        "versions": versions, "hardware": hardware,
                        "run_fingerprint": fingerprint,
                        "checkpoint_migration": migration})
        atomic_write_bytes(history_path, json.dumps(history, indent=2).encode())

        metadata = {"run_fingerprint": fingerprint,
                    "resume_key_scope": "protocol only; accelerator, driver "
                                        "and framework build excluded",
                    "sessions": len(history),
                    "protocol_versions": protocol_versions,
                    "environment_history": "environment_history.json",
                    "classical_fingerprint": classical_marker["run_fingerprint"],
                    "manifest_sha256": manifest_sha,
                    "private_zip_sha256": classical_meta["private_zip_sha256"],
                    "classical_zip_sha256": sha256(classical_zip.read_bytes()),
                    "pilot_not_publishable": pilot,
                    "model_selection": "separate masked validation partition",
                    "train_and_val_data": "condition-specific masks, no test labels",
                    "window_overlap_aggregation": "arithmetic mean",
                    "config": config, "versions": versions,
                    "hardware": hardware}
        atomic_write_bytes(output_dir / "run_metadata.json",
                           json.dumps(metadata, indent=2).encode())

        records = receipt["assets"][:1] if pilot else receipt["assets"]
        folds = ("A",) if pilot else tuple(FOLDS)
        conditions = [(b, r, s) for b in BLOCKS for r in RATES for s in SEEDS]
        if pilot:
            conditions = conditions[:1]
        expected_shards = len(records) * len(folds) * len(conditions)
        session_started = time.time()
        session_done = 0
        already = sum(1 for _ in shard_dir.glob("*.csv"))
        print(f"Resuming: {already}/{expected_shards} shards already on disk "
              f"({100 * already / expected_shards:.1f}%)", flush=True)
        for asset_index, record in enumerate(records, 1):
            ticker = record["ticker"]
            frame = load_verified_asset(private, record)
            for fold in folds:
                reference = verified_classical_shard(
                    classical, ticker, fold, classical_marker["run_fingerprint"])
                partitions = split_frame(frame, fold)
                train = partitions["train"].to_numpy(float)
                val = partitions["val"].to_numpy(float)
                test = partitions["test"].to_numpy(float)
                for condition_index, (block, rate, repetition_seed) in enumerate(conditions, 1):
                    stem = f"{ticker}_{fold}_b{block}_r{round(rate*100)}_s{repetition_seed}"
                    csv_path = shard_dir / (stem + ".csv")
                    meta_path = shard_dir / (stem + ".meta.json")
                    val_csv_path = validation_dir / (stem + ".csv")
                    val_meta_path = validation_dir / (stem + ".meta.json")
                    if (csv_path.exists() and meta_path.exists() and
                            val_csv_path.exists() and val_meta_path.exists()):
                        sidecar = json.loads(meta_path.read_text())
                        val_sidecar = json.loads(val_meta_path.read_text())
                        if (sidecar.get("run_fingerprint") == fingerprint and
                                sidecar.get("expected_rows") == 10 and
                                sidecar.get("csv_sha256") == sha256(csv_path.read_bytes()) and
                                val_sidecar.get("run_fingerprint") == fingerprint and
                                val_sidecar.get("expected_rows") == 35 and
                                val_sidecar.get("csv_sha256") == sha256(val_csv_path.read_bytes())):
                            continue
                    train_seed = stable_mask_seed(manifest_sha, ticker, fold, "train",
                                                  block, rate, repetition_seed)
                    val_seed = stable_mask_seed(manifest_sha, ticker, fold, "val",
                                                block, rate, repetition_seed)
                    test_seed = stable_mask_seed(manifest_sha, ticker, fold, "test",
                                                 block, rate, repetition_seed)
                    if reference[(block, rate, repetition_seed)] != (train_seed, test_seed):
                        raise AssertionError("neural and classical mask seeds differ")
                    train_hidden = strict_block_mask(train.shape, rate, block,
                                                     train_seed, EDGE_GUARD)
                    val_hidden = strict_block_mask(val.shape, rate, block,
                                                   val_seed, EDGE_GUARD)
                    test_hidden = strict_block_mask(test.shape, rate, block,
                                                    test_seed, EDGE_GUARD)
                    mean, scale = train_observed_scaler(train, train_hidden)
                    train_input = corrupt((train - mean) / scale, train_hidden)
                    val_truth = (val - mean) / scale
                    val_input = corrupt(val_truth, val_hidden)
                    test_input = corrupt((test - mean) / scale, test_hidden)
                    lengths = run_lengths(test_hidden)
                    if int(lengths.max()) > block:
                        raise AssertionError("test mask exceeds requested run length")
                    common = {"ticker": ticker, "country": record["country"],
                              "region": record["region"], "fold": fold,
                              "test_start_year": FOLDS[fold]["test"][0],
                              "test_end_year": FOLDS[fold]["test"][1],
                              "block_length": block, "rate": rate,
                              "seed": repetition_seed,
                              "train_mask_seed": train_seed,
                              "val_mask_seed": val_seed,
                              "test_mask_seed": test_seed,
                              "actual_max_test_run": int(lengths.max())}
                    rows = []
                    val_common = {"ticker": ticker, "country": record["country"],
                                  "region": record["region"], "fold": fold,
                                  "block_length": block, "rate": rate,
                                  "seed": repetition_seed,
                                  "train_mask_seed": train_seed,
                                  "val_mask_seed": val_seed,
                                  "partition": "validation"}
                    val_rows = []
                    for method, (val_estimate, val_elapsed, val_warnings) in run_methods(
                            train_input, val_input, repetition_seed).items():
                        val_rows.extend(score_features(
                            val, val_estimate, val_hidden, mean, scale,
                            {**val_common, "method": method},
                            val_elapsed, val_warnings))
                    for method in NEURAL_METHODS:
                        model_seed = training_seed(manifest_sha, ticker, fold,
                                                   block, rate, repetition_seed, method)
                        val_estimate, estimate, elapsed = fit_and_predict(
                            method, train_input, val_input, val_truth,
                            test_input, model_seed, device,
                            epochs=1 if pilot else EPOCHS)
                        val_rows.extend(score_features(
                            val, val_estimate, val_hidden, mean, scale,
                            {**val_common, "method": method,
                             "model_seed": model_seed}, elapsed, 0))
                        scored = score_features(test, estimate, test_hidden,
                                                mean, scale,
                                                {**common, "method": method,
                                                 "model_seed": model_seed,
                                                 "window_length": N_STEPS,
                                                 "window_stride": STRIDE,
                                                 "epochs_max": 1 if pilot else EPOCHS,
                                                 "patience": 1 if pilot else PATIENCE},
                                                elapsed, 0)
                        rows.extend(scored)
                    if len(rows) != len(NEURAL_METHODS) * len(FEATURES):
                        raise AssertionError("neural condition row count mismatch")
                    if len(val_rows) != (len(METHODS) + len(NEURAL_METHODS)) * len(FEATURES):
                        raise AssertionError("validation condition row count mismatch")
                    val_data = pd.DataFrame(val_rows).to_csv(
                        index=False, float_format="%.17g").encode()
                    atomic_write_bytes(val_csv_path, val_data)
                    val_sidecar = {"run_fingerprint": fingerprint,
                                   "expected_rows": len(val_rows),
                                   "csv_sha256": sha256(val_data),
                                   "pilot_not_publishable": pilot}
                    atomic_write_bytes(val_meta_path,
                                       json.dumps(val_sidecar, indent=2).encode())
                    data = pd.DataFrame(rows).to_csv(
                        index=False, float_format="%.17g").encode()
                    atomic_write_bytes(csv_path, data)
                    sidecar = {"run_fingerprint": fingerprint,
                               "expected_rows": len(rows),
                               "csv_sha256": sha256(data),
                               "pilot_not_publishable": pilot}
                    atomic_write_bytes(meta_path, json.dumps(sidecar, indent=2).encode())
                    session_done += 1
                    if session_done % 10 == 0:
                        rate_s = (time.time() - session_started) / session_done
                        left = expected_shards - already - session_done
                        print(f"[{asset_index}/{len(records)}] {ticker}/{fold} "
                              f"{condition_index}/{len(conditions)} | "
                              f"toplam {already + session_done}/{expected_shards} "
                              f"(%{100 * (already + session_done) / expected_shards:.1f}) | "
                              f"{rate_s:.0f} s/shard | kalan ~{rate_s * left / 3600:.1f} saat",
                              flush=True)

        verified = 0
        for record in records:
            for fold in folds:
                for block, rate, repetition_seed in conditions:
                    stem = (f"{record['ticker']}_{fold}_b{block}_"
                            f"r{round(rate*100)}_s{repetition_seed}")
                    csv_path = shard_dir / (stem + ".csv")
                    sidecar = json.loads((shard_dir / (stem + ".meta.json")).read_text())
                    val_csv_path = validation_dir / (stem + ".csv")
                    val_sidecar = json.loads(
                        (validation_dir / (stem + ".meta.json")).read_text())
                    if (sidecar["run_fingerprint"] != fingerprint or
                            sidecar["csv_sha256"] != sha256(csv_path.read_bytes()) or
                            val_sidecar["run_fingerprint"] != fingerprint or
                            val_sidecar["csv_sha256"] != sha256(val_csv_path.read_bytes())):
                        raise AssertionError(f"invalid neural checkpoint: {stem}")
                    verified += 1
        if verified != expected_shards:
            raise AssertionError("missing neural result shards")
        marker = {"expected_shards": expected_shards,
                  "expected_rows": expected_shards * len(NEURAL_METHODS) * len(FEATURES),
                  "expected_validation_rows": expected_shards *
                  (len(METHODS) + len(NEURAL_METHODS)) * len(FEATURES),
                  "pilot_not_publishable": pilot,
                  "run_fingerprint": fingerprint}
        atomic_write_bytes(output_dir / "COMPLETE.marker",
                           json.dumps(marker, indent=2).encode())
        print("COMPLETE", output_dir, flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--snapshot", type=Path, required=True)
    parser.add_argument("--classical", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--pilot", action="store_true",
                        help="one asset/fold/condition and one epoch; not publishable")
    args = parser.parse_args()
    run(args.snapshot, args.classical, args.out, pilot=args.pilot)


In [ ]:
%%writefile /content/itri_neural/analyze_all_benchmarks.py
#!/usr/bin/env python3
"""Audit and combine the frozen classical and BRITS/SAITS external results.

The primary endpoint is a feature-macro normalized RMSE for each asset/fold/
condition/method. Asset summaries then average folds and mask conditions;
the 5,508 conditions are not treated as independent market replicates.
"""

from __future__ import annotations

import argparse
import hashlib
import io
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd


CLASSICAL = ("linear", "pchip", "knn5", "chained_ridge", "itri_w1_p5")
NEURAL = ("brits", "saits")
ALL_METHODS = CLASSICAL + NEURAL
KEY = ["ticker", "fold", "block_length", "rate", "seed", "feature", "method"]
CONDITION = ["ticker", "country", "region", "fold", "block_length",
             "rate", "seed", "method"]
NUMERIC = ["mae_raw", "rmse_raw", "mae_trainstd", "rmse_trainstd",
           "fit_and_impute_seconds"]


def sha256(blob: bytes) -> str:
    return hashlib.sha256(blob).hexdigest()


def markdown_table(frame: pd.DataFrame) -> str:
    def cell(value: object) -> str:
        if isinstance(value, (float, np.floating)):
            return f"{value:.6f}"
        return str(value)

    header = "| " + " | ".join(frame.columns) + " |"
    rule = "| " + " | ".join("---" for _ in frame.columns) + " |"
    rows = ["| " + " | ".join(cell(value) for value in row) + " |"
            for row in frame.itertuples(index=False, name=None)]
    return "\n".join([header, rule, *rows])


def load_checked(source: Path, expected_methods: tuple[str, ...]) -> tuple[pd.DataFrame, dict, dict]:
    if source.is_dir():
        names = sorted(p.name for p in (source / "shards").glob("*.csv"))

        def read(name: str) -> bytes:
            return (source / name).read_bytes()
    elif zipfile.is_zipfile(source):
        archive = zipfile.ZipFile(source)
        if archive.testzip() is not None:
            raise ValueError(f"corrupt ZIP: {source}")
        names = sorted(Path(n).name for n in archive.namelist()
                       if n.startswith("results/shards/") and n.endswith(".csv"))

        def read(name: str) -> bytes:
            return archive.read("results/" + name)
    else:
        raise ValueError(f"expected result directory or ZIP: {source}")
    marker = json.loads(read("COMPLETE.marker"))
    metadata = json.loads(read("run_metadata.json"))
    if marker["pilot_not_publishable"] or metadata["pilot_not_publishable"]:
        raise ValueError("pilot output is not a full scientific result")
    if (len(names) != marker["expected_shards"] or
            marker["run_fingerprint"] != metadata["run_fingerprint"]):
        raise ValueError("incomplete or mixed result checkpoints")
    frames = []
    for name in names:
        blob = read("shards/" + name)
        sidecar = json.loads(read("shards/" + name.replace(".csv", ".meta.json")))
        if (sha256(blob) != sidecar["csv_sha256"] or
                sidecar["run_fingerprint"] != marker["run_fingerprint"]):
            raise ValueError(f"checksum/fingerprint mismatch: {name}")
        frame = pd.read_csv(io.BytesIO(blob), float_precision="round_trip")
        if len(frame) != sidecar["expected_rows"]:
            raise ValueError(f"row count mismatch: {name}")
        frames.append(frame)
    data = pd.concat(frames, ignore_index=True)
    if len(data) != marker["expected_rows"] or set(data.method) != set(expected_methods):
        raise ValueError("result row count or method registry mismatch")
    if data.duplicated(KEY).any() or data[NUMERIC].isna().any().any():
        raise ValueError("duplicate key or missing numeric result")
    if not np.isfinite(data[NUMERIC].to_numpy(float)).all():
        raise ValueError("non-finite numeric result")
    if (data.masked_cells <= 0).any() or (data[NUMERIC] < 0).any().any():
        raise ValueError("invalid masked count, error or runtime")
    if source.is_file():
        archive.close()
    return data, marker, metadata


def load_validation_checked(source: Path, marker: dict, metadata: dict) -> pd.DataFrame:
    if source.is_dir():
        names = sorted(p.name for p in (source / "validation_shards").glob("*.csv"))

        def read(name: str) -> bytes:
            return (source / "validation_shards" / name).read_bytes()
    else:
        archive = zipfile.ZipFile(source)
        names = sorted(Path(n).name for n in archive.namelist()
                       if n.startswith("results/validation_shards/") and n.endswith(".csv"))

        def read(name: str) -> bytes:
            return archive.read("results/validation_shards/" + name)
    if len(names) != marker["expected_shards"]:
        raise ValueError("validation shard grid is incomplete")
    frames = []
    for name in names:
        blob = read(name)
        sidecar = json.loads(read(name.replace(".csv", ".meta.json")))
        if (sidecar["run_fingerprint"] != metadata["run_fingerprint"] or
                sidecar["csv_sha256"] != sha256(blob) or
                sidecar["expected_rows"] != 35):
            raise ValueError(f"invalid validation checkpoint: {name}")
        frame = pd.read_csv(io.BytesIO(blob), float_precision="round_trip")
        if len(frame) != 35:
            raise ValueError(f"validation row count mismatch: {name}")
        frames.append(frame)
    if source.is_file():
        archive.close()
    data = pd.concat(frames, ignore_index=True)
    if (len(data) != marker["expected_validation_rows"] or
            set(data.method) != set(ALL_METHODS) or
            data.duplicated(KEY).any() or
            (data.masked_cells <= 0).any() or
            not np.isfinite(data[NUMERIC].to_numpy(float)).all()):
        raise ValueError("invalid complete validation grid")
    return data


def reconcile(classical: pd.DataFrame, neural: pd.DataFrame,
              classical_meta: dict, neural_meta: dict) -> pd.DataFrame:
    if (classical_meta["run_fingerprint"] != neural_meta["classical_fingerprint"] or
            classical_meta["manifest_sha256"] != neural_meta["manifest_sha256"] or
            classical_meta["private_zip_sha256"] != neural_meta["private_zip_sha256"]):
        raise ValueError("classical/neural input provenance does not match")
    data = pd.concat([classical, neural], ignore_index=True, sort=False)
    if len(data) != 34 * 3 * 6 * 3 * 3 * len(ALL_METHODS) * 5:
        raise ValueError("combined prespecified grid is incomplete")
    if data.duplicated(KEY).any():
        raise ValueError("duplicate combined result key")
    condition_keys = ["ticker", "fold", "block_length", "rate", "seed"]
    methods = data.groupby(condition_keys)["method"].nunique()
    if len(methods) != 34 * 3 * 6 * 3 * 3 or not methods.eq(len(ALL_METHODS)).all():
        raise ValueError("at least one condition lacks a comparator")
    feature_counts = data.groupby(condition_keys + ["method"])["feature"].nunique()
    if not feature_counts.eq(5).all():
        raise ValueError("at least one method lacks a feature")
    for field in ("train_mask_seed", "test_mask_seed", "masked_cells",
                  "actual_max_test_run"):
        varying = data.groupby(condition_keys + ["feature"])[field].nunique()
        if not varying.eq(1).all():
            raise ValueError(f"methods did not share {field}")
    return data


def summarize(data: pd.DataFrame, validation: pd.DataFrame, output_dir: Path,
              classical_meta: dict, neural_meta: dict) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)
    data.sort_values(KEY).to_csv(output_dir / "feature_metrics_combined.csv",
                                 index=False, float_format="%.17g")
    condition = data.groupby(CONDITION, as_index=False).agg(
        nrmse=("rmse_trainstd", "mean"),
        nmae=("mae_trainstd", "mean"),
        masked_cells_total=("masked_cells", "sum"),
        fit_and_impute_seconds=("fit_and_impute_seconds", "first"),
    )
    condition.to_csv(output_dir / "condition_macro.csv", index=False,
                     float_format="%.17g")
    validation.to_csv(output_dir / "validation_feature_metrics.csv", index=False,
                      float_format="%.17g")
    val_condition = validation.groupby(CONDITION, as_index=False).agg(
        validation_nrmse=("rmse_trainstd", "mean"),
        validation_nmae=("mae_trainstd", "mean"))
    paired_conditions = condition.merge(
        val_condition,
        on=["ticker", "country", "region", "fold", "block_length", "rate",
            "seed", "method"], validate="one_to_one")
    if len(paired_conditions) != len(condition):
        raise ValueError("validation/test condition keys differ")
    paired_conditions["test_minus_validation_nrmse"] = (
        paired_conditions.nrmse - paired_conditions.validation_nrmse)
    paired_conditions.to_csv(output_dir / "validation_test_conditions.csv",
                             index=False, float_format="%.17g")
    gap_asset = paired_conditions.groupby(["ticker", "method"],
                                          as_index=False).agg(
        mean_test_nrmse=("nrmse", "mean"),
        mean_validation_nrmse=("validation_nrmse", "mean"),
        mean_test_minus_validation_nrmse=("test_minus_validation_nrmse", "mean"))
    gap_asset.to_csv(output_dir / "validation_test_asset.csv", index=False,
                     float_format="%.17g")
    gap_summary = gap_asset.groupby("method", as_index=False).agg(
        mean_test_nrmse=("mean_test_nrmse", "mean"),
        mean_validation_nrmse=("mean_validation_nrmse", "mean"),
        mean_test_minus_validation_nrmse=("mean_test_minus_validation_nrmse", "mean"))
    gap_summary.to_csv(output_dir / "validation_test_method.csv", index=False,
                       float_format="%.17g")
    asset = condition.groupby(["ticker", "country", "region", "method"],
                              as_index=False).agg(
        nrmse=("nrmse", "mean"), nmae=("nmae", "mean"))
    asset.to_csv(output_dir / "asset_method.csv", index=False,
                 float_format="%.17g")
    block_asset = condition.groupby(["ticker", "region", "block_length", "method"],
                                    as_index=False).agg(nrmse=("nrmse", "mean"),
                                                        nmae=("nmae", "mean"))
    block_summary = block_asset.groupby(["block_length", "method"],
                                        as_index=False).agg(
        mean_asset_nrmse=("nrmse", "mean"),
        median_asset_nrmse=("nrmse", "median"),
        mean_asset_nmae=("nmae", "mean"))
    block_summary.to_csv(output_dir / "block_method.csv", index=False,
                         float_format="%.17g")
    block_rate_asset = condition.groupby(
        ["ticker", "block_length", "rate", "method"], as_index=False).agg(
            nrmse=("nrmse", "mean"), nmae=("nmae", "mean"))
    block_rate_summary = block_rate_asset.groupby(
        ["block_length", "rate", "method"], as_index=False).agg(
            mean_asset_nrmse=("nrmse", "mean"),
            median_asset_nrmse=("nrmse", "median"),
            mean_asset_nmae=("nmae", "mean"))
    block_rate_summary.to_csv(output_dir / "block_rate_method.csv", index=False,
                              float_format="%.17g")
    feature_asset = data.groupby(["ticker", "feature", "method"],
                                 as_index=False).agg(
        nrmse=("rmse_trainstd", "mean"), nmae=("mae_trainstd", "mean"))
    feature_asset.groupby(["feature", "method"], as_index=False).agg(
        mean_asset_nrmse=("nrmse", "mean"),
        mean_asset_nmae=("nmae", "mean")).to_csv(
            output_dir / "feature_method.csv", index=False, float_format="%.17g")
    fold_asset = condition.groupby(["ticker", "fold", "method"],
                                   as_index=False).agg(nrmse=("nrmse", "mean"))
    fold_asset.groupby(["fold", "method"], as_index=False).agg(
        mean_asset_nrmse=("nrmse", "mean"),
        median_asset_nrmse=("nrmse", "median")).to_csv(
            output_dir / "fold_method.csv", index=False, float_format="%.17g")
    short = condition.loc[condition.block_length.isin((1, 2))]
    short_asset = short.groupby(["ticker", "method"], as_index=False).agg(
        nrmse=("nrmse", "mean"), nmae=("nmae", "mean"))
    short_summary = short_asset.groupby("method", as_index=False).agg(
        mean_asset_nrmse=("nrmse", "mean"),
        median_asset_nrmse=("nrmse", "median"),
        mean_asset_nmae=("nmae", "mean"))
    short_summary.to_csv(output_dir / "short_gap_method.csv", index=False,
                         float_format="%.17g")
    all_wide = asset.pivot(index="ticker", columns="method", values="nrmse")
    short_wide = short_asset.pivot(index="ticker", columns="method", values="nrmse")
    paired = []
    for scope, wide in (("all_blocks", all_wide), ("blocks_1_2", short_wide)):
        for ticker, row in wide.iterrows():
            for method in ALL_METHODS:
                if method != "itri_w1_p5":
                    paired.append({"scope": scope, "ticker": ticker,
                                   "comparator": method,
                                   "itri_nrmse": row["itri_w1_p5"],
                                   "comparator_nrmse": row[method],
                                   "itri_minus_comparator": row["itri_w1_p5"] - row[method]})
    paired_frame = pd.DataFrame(paired)
    paired_frame.to_csv(output_dir / "paired_asset_differences.csv", index=False,
                        float_format="%.17g")
    win_summary = paired_frame.groupby(["scope", "comparator"], as_index=False).agg(
        itri_wins=("itri_minus_comparator", lambda x: int((x < 0).sum())),
        itri_losses=("itri_minus_comparator", lambda x: int((x > 0).sum())),
        mean_difference=("itri_minus_comparator", "mean"),
        median_difference=("itri_minus_comparator", "median"))
    win_summary.to_csv(output_dir / "paired_win_counts.csv", index=False,
                       float_format="%.17g")
    region_asset = asset.groupby(["region", "method"], as_index=False).agg(
        mean_asset_nrmse=("nrmse", "mean"), asset_count=("ticker", "nunique"))
    region_asset.to_csv(output_dir / "region_method.csv", index=False,
                        float_format="%.17g")
    runtime = condition.groupby("method", as_index=False).agg(
        total_fit_and_impute_seconds=("fit_and_impute_seconds", "sum"),
        median_condition_seconds=("fit_and_impute_seconds", "median"))
    runtime.to_csv(output_dir / "runtime_method.csv", index=False,
                   float_format="%.17g")
    overview = asset.groupby("method", as_index=False).agg(
        mean_asset_nrmse=("nrmse", "mean"),
        median_asset_nrmse=("nrmse", "median"),
        mean_asset_nmae=("nmae", "mean"))
    overview.to_csv(output_dir / "overall_method.csv", index=False,
                    float_format="%.17g")
    audit = {"status": "PASS", "assets": data.ticker.nunique(),
             "folds": sorted(data.fold.unique().tolist()),
             "methods": list(ALL_METHODS),
             "feature_rows": len(data), "condition_method_rows": len(condition),
             "asset_method_rows": len(asset),
             "validation_feature_rows": len(validation),
             "classical_fingerprint": classical_meta["run_fingerprint"],
             "neural_fingerprint": neural_meta["run_fingerprint"],
             "unit_of_analysis": "asset (conditions and folds averaged within asset)",
             "inferential_tests": "none; ETF dependence not assumed away"}
    (output_dir / "analysis_audit.json").write_text(
        json.dumps(audit, indent=2), encoding="utf-8")
    lines = ["# External-validation results (descriptive)", "",
             "The primary endpoint is feature-macro NRMSE. Results below first "
             "average conditions and folds within each asset, then summarize "
             "the 34 assets. They do not treat masks or ETFs as independent markets.",
             "", "## All block lengths", "",
             markdown_table(overview.sort_values("mean_asset_nrmse")),
             "", "## One- and two-day blocks", "",
             markdown_table(short_summary.sort_values("mean_asset_nrmse")),
             "", "## Paired asset win counts", "",
             markdown_table(win_summary),
             "", "## Validation–test diagnostic", "",
             "Validation selected neural epochs and is not a second held-out test.",
             "", markdown_table(gap_summary),
             "", "No causal or population-level significance claim follows "
             "from these descriptive comparisons."]
    (output_dir / "paper_tables.md").write_text("\n".join(lines) + "\n",
                                                 encoding="utf-8")
    return audit


def run(classical_source: Path, neural_source: Path, output_dir: Path) -> dict:
    if not classical_source.is_file():
        raise ValueError("classical results must be the exact original ZIP")
    classical, _, classical_meta = load_checked(classical_source, CLASSICAL)
    neural, neural_marker, neural_meta = load_checked(neural_source, NEURAL)
    validation = load_validation_checked(neural_source, neural_marker, neural_meta)
    if neural_meta["classical_zip_sha256"] != sha256(classical_source.read_bytes()):
        raise ValueError("neural experiment used a different classical result ZIP")
    data = reconcile(classical, neural, classical_meta, neural_meta)
    return summarize(data, validation, output_dir, classical_meta, neural_meta)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--classical", type=Path, required=True)
    parser.add_argument("--neural", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    args = parser.parse_args()
    print(json.dumps(run(args.classical, args.neural, args.out), indent=2))


In [ ]:
%%writefile /content/itri_neural/NEURAL_PROTOCOL_v1.md
# ITRI external validation: BRITS/SAITS extension, frozen 2026-09-23

This implementation plan was fixed **after** the classical results were
inspected but **before** BRITS/SAITS external results were generated. It is
not a blinded preregistration. Do not adjust architecture, epochs, windows,
or exclusions in response to held-out neural scores without labeling a new
exploratory analysis.

## Data and evaluation

- Use the same audited private 34-ETF snapshot, OHLCV features, chronological
  A/B/C folds, six block lengths, three missing rates, three mask seeds,
  edge guard, and strict mask generator as the completed classical run.
- For every asset × fold × block × rate × seed, fit each model separately on
  the corresponding masked training partition. No test-period value or mask
  is used in optimization or epoch selection. Validation-period ground truth
  is used only for checkpoint selection; test ground truth is used only to
  calculate final metrics.
- Compute each feature's mean and standard deviation from initially observed
  **training** cells for the matching condition, exactly as in the classical
  benchmark. Apply that scale to train, validation and test. No future asset
  data are pooled across chronological folds.
- Training, validation and test windows have 64 trading days and stride 16.
  Add a final window ending at the partition end so every day is covered.
  Average overlapping test-window estimates cellwise, then restore all
  originally observed test cells before scoring. No window crosses a
  train/validation/test partition boundary.
- BRITS and SAITS each receive 30 maximum epochs, early-stopping patience 5,
  batch size 32, and validation MSE as supplied by PyPOTS 1.5. BRITS has
  recurrent hidden size 64. SAITS uses two layers, model width 64, four heads,
  key/value width 16, feed-forward width 128 and dropout 0.1. SAITS retains
  its library-default internal 20% masked-imputation-training augmentation;
  that self-supervised augmentation is distinct from the fixed test mask.
- Initialization/training seeds are stable hashes of manifest, asset, fold,
  condition and method. Python, NumPy and Torch are seeded. CuDNN
  deterministic mode is enabled, but bitwise equality across GPU hardware,
  drivers, Python and Torch versions is **not** claimed. Exact versions and
  hardware must be reported with results.
- Each test condition yields the same five raw MAE/RMSE and training-standardized
  MAE/RMSE measurements as the classical code. The primary endpoint is the
  macro mean of five feature-wise standardized RMSE values. Conditions and
  folds are averaged within each asset; 5,508 masks are not independent
  market replicates. Regional/shared-shock dependence remains a limitation.
- Validation diagnostics are recorded separately for all seven methods. The
  same condition-specific masked training input is used to fit each classical
  validation comparator; the neural validation scores are calculated from
  the validation period used for epoch selection. Therefore validation scores
  are descriptive model-selection diagnostics, **not** a second held-out test.
- Runtime includes neural fitting, validation imputation and test imputation
  for a condition, while
  classical runtime uses each method's existing implementation. These costs
  are reported descriptively, not claimed as perfectly hardware-matched.

## Computation and provenance

- The grid requires 34 × 3 × 6 × 3 × 3 × 2 = **11,016 neural fits**. A Colab
  session may disconnect before completion. Result shards are checkpointed
  after every condition to Google Drive and verified by SHA-256 and a
  code/environment fingerprint on resumption. Re-run the notebook after a
  disconnection; do not report a partial result directory.
- The full-run marker is emitted only after all 5,508 condition shards and
  55,080 neural test-feature rows and 192,780 validation-feature rows are
  present and verified. The combined test audit must then reconcile neural
  rows with 137,700 classical rows to 192,780 test-feature rows.
- The public result archive contains metrics, protocol, versions, code and
  input hashes, but not vendor OHLCV or model weights. Keep the private
  snapshot outside the public repository unless its license permits release.
- A one-condition/one-epoch pilot checks code execution only. Pilot metrics
  are explicitly not publishable and must not enter manuscript tables.

Official implementation/API references:

- BRITS: https://docs.pypots.com/en/latest/_modules/pypots/imputation/brits/model.html
- SAITS: https://docs.pypots.com/en/latest/_modules/pypots/imputation/saits/model.html
- PyPOTS package/version: https://pypi.org/project/pypots/


## File 1/2 — private data snapshot

Select **`ITRI_external_PRIVATE_data_snapshot.zip`** from your computer. Select this file only. It is the original audited data archive, not the QC receipt.

In [ ]:
import hashlib
from google.colab import files

print('FILE 1/2: Select ITRI_external_PRIVATE_data_snapshot.zip only.', flush=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Select exactly one file: ITRI_external_PRIVATE_data_snapshot.zip')
filename, content = next(iter(uploaded.items()))
if hashlib.sha256(content).hexdigest() != '48dc63654f63db3ef9d6d5db7ab69ad502aa079e9f6deb4b276cf9b1170f324b':
    raise ValueError(f'Wrong file selected: {filename}. Expected ITRI_external_PRIVATE_data_snapshot.zip')
snapshot_path = work_dir / 'PRIVATE_vendor_snapshot.zip'
snapshot_path.write_bytes(content)
del uploaded, content
print('Private data snapshot verified; raw OHLCV will stay outside Drive output.')


## File 2/2 — completed classical results

Select **`ITRI_classical_full_results.zip`** from your computer. Select this file only. Do not select an incomplete classical run. The two files may be in different folders.

In [ ]:
print('FILE 2/2: Select ITRI_classical_full_results.zip only.', flush=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Select exactly one file: ITRI_classical_full_results.zip')
filename, content = next(iter(uploaded.items()))
if hashlib.sha256(content).hexdigest() != '83b0c3959bb4d65a3a0ed09c30e50c76635879c69314488b7e768308af3d05f4':
    raise ValueError(f'Wrong file selected: {filename}. Expected ITRI_classical_full_results.zip')
classical_path = work_dir / 'ITRI_classical_full_results.zip'
classical_path.write_bytes(content)
del uploaded, content
print('Classical results verified; both input archives are ready.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
result_dir = Path('/content/drive/MyDrive/ITRI_neural_external_validation')
result_dir.mkdir(parents=True, exist_ok=True)
print('Resumable metric checkpoints:', result_dir)

## Durum

Drive'daki ilerlemeyi gosterir. Farkli bir GPU ya da farkli bir torch surumu sorun degildir: tamamlanmis shard'lar SHA-256 ile dogrulanip yeni anahtara tasinir, calisma kaldigi yerden devam eder. Her oturumun donanim ve kutuphane bilgisi `environment_history.json` dosyasina yazilir ve makalede bu sekilde raporlanmalidir.


In [ ]:
# Durum ozeti. Ortam farkli olsa bile calismayi DURDURMAZ:
# tamamlanmis shard'lar yeni anahtara tasinir, kalanindan devam edilir.
import json, torch, platform
from pathlib import Path

shards = result_dir / 'shards'
done = len(list(shards.glob('*.csv'))) if shards.is_dir() else 0
print(f"Drive'da tamamlanmis shard: {done}/5508  (%{100*done/5508:.1f})")
print('Bu oturumun GPU su:', torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else 'CPU (yavas olur, GPU runtime secin)')
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
hist = result_dir / 'environment_history.json'
if hist.exists():
    print('Onceki oturum sayisi:', len(json.loads(hist.read_text())))


In [ ]:
import importlib
import sys

sys.path.insert(0, str(work_dir))
importlib.invalidate_caches()
import masking
importlib.reload(masking)
import run_classical_benchmark
importlib.reload(run_classical_benchmark)
import run_neural_benchmark
importlib.reload(run_neural_benchmark)
run_neural_benchmark.run(snapshot_path, classical_path, result_dir)


In [ ]:
import importlib
import analyze_all_benchmarks
importlib.reload(analyze_all_benchmarks)

analysis_dir = work_dir / 'analysis'
audit = analyze_all_benchmarks.run(classical_path, result_dir, analysis_dir)
print('Combined audit:', audit)


In [ ]:
import zipfile

result_zip = Path('/content/ITRI_all_methods_results.zip')
with zipfile.ZipFile(result_zip, 'w', compression=zipfile.ZIP_DEFLATED) as out:
    for path in result_dir.rglob('*'):
        if path.is_file():
            out.write(path, 'neural_results/' + str(path.relative_to(result_dir)))
    for path in analysis_dir.rglob('*'):
        if path.is_file():
            out.write(path, 'analysis/' + str(path.relative_to(analysis_dir)))
    for name in ('masking.py', 'run_classical_benchmark.py',
                 'run_neural_benchmark.py', 'analyze_all_benchmarks.py',
                 'NEURAL_PROTOCOL_v1.md'):
        out.write(work_dir / name, 'code_and_protocol/' + name)
    out.write(classical_path, 'classical_results/ITRI_classical_full_results.zip')
print('Complete:', result_zip.name, '— downloading now')
files.download(str(result_zip))
